In [1]:
import torch
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM
from data_utils import *
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from icecream import ic
import torch.nn.functional as F
from tqdm.notebook import tqdm

## tokenizer

In [2]:
from transformers import AutoTokenizer
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
REWARD_TOKEN_ID = tokenizer.eos_token_id
tokenizer.add_special_tokens({
    "pad_token": "[PAD]"
})

# IMPORTANT: usually the exisitng works use pad from left but acc to the N+ paper implemenation we should pad from righ
tokenizer.padding_side = "right"

## dataloader

In [3]:
from datasets import load_dataset
ds = load_dataset("vwxyzjn/summarize_from_feedback_oai_preprocessing_1706381144")

In [4]:
from datasets import load_dataset
ds = load_dataset("vwxyzjn/summarize_from_feedback_oai_preprocessing_1706381144")
batch_size = 8
train_dataset = PreferenceDataset(
    ds['train'],
    tokenizer
)
train_loader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    shuffle = True,
    collate_fn = preference_collate_fn
)


eval_dataset = PreferenceDataset(
    ds['validation'],
    tokenizer
)
eval_loader = DataLoader(
    eval_dataset,
    batch_size = batch_size,
    shuffle = True,
    collate_fn = preference_collate_fn
)



In [5]:
# ds['train']['query']a

In [6]:
ds['train'][0]['summaries'][0]['text']

' Mum is mad at me for not flying on my own trip to meet my boyfriend.'

In [7]:
print(train_dataset[0].keys())

dict_keys(['query_token', 'chosen_token', 'rejected_token', 'query_chosen_token', 'query_rejected_token', 'query_chosen_attention_mask', 'query_rejected_attention_mask', 'choice'])


/media/system/ZERBUIS_EXT_STOR/temp/exp/rlhf/data_utils.py:190: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "query_token": torch.tensor(
/media/system/ZERBUIS_EXT_STOR/temp/exp/rlhf/data_utils.py:195: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "chosen_token": torch.tensor(
/media/system/ZERBUIS_EXT_STOR/temp/exp/rlhf/data_utils.py:200: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "rejected_token": torch.tensor(
/media/system/ZERBUIS_EXT_STOR/temp/exp/rlhf/data_utils.py:205: UserWarning: To copy construct from a tensor, it is recomme

In [8]:
batch = next(iter(train_loader))
print("")

In [9]:
tokenizer.pad_token_id

50257

In [10]:
# ic(ds['train'][0]['query'])
# ic(ds['train'][0]['summaries'])
# print("")

In [11]:
batch['query_chosen_token'][1][537] #[torch.where(batch['query_chosen_attention_mask'][0] == 1)[0][-1]]

tensor(50257)

In [12]:
batch['query_chosen_attention_mask'].shape

torch.Size([8, 565])

In [13]:
batch['query_chosen_attention_mask'].sum(dim=1) - 1


tensor([325, 441, 370, 442, 396, 462, 261, 400])

In [14]:
batch['query_chosen_token'][0][batch['query_chosen_token'][0] != tokenizer.pad_token_id]

tensor([   50, 10526, 22083, 49828,    25,   374,    14, 24546,   278,   198,
          198, 49560,  2538,    25,  1160, 42635,  1468,  1995,   290,  3656,
          287,   257,  5802,  4136,    12,   815,   314, 33858,    30,   198,
          198, 32782,    25, 14690,   612,     0,  1406,   284,  1577,   345,
          617,  4469,    11,   314,  1392,  6405,   379,  1248,   284,   616,
         5229,   508,   373,  1987,    13,   775,   550,   257,  5156,   767,
         1933,  2084,   290,   314,  1053,   587, 10589,  1363,   351,   607,
           11,   543,   314,  1842,    13,   632,   338,   257,  1342,   621,
         7306,  3074,   780,   616,  5229,   468,   284,   670,  3126,   284,
         4317,  2250,   257,  1285,    13,   887,   314,   836,   470,   765,
          616,  4957,   287,  1110,  6651,    13,  2102,    11,   379,   428,
          966,    11,   356,   389,  8523,  1972,   416,    11,   314,  1239,
          766,   616,  5229,   357,   258,  2499,   362,   358, 

In [15]:
len(np.where(batch['query_chosen_token'][0] != tokenizer.pad_token_id)[0])

326

## reward model

In [ ]:
sft_model_path = 'sft_epoch1_dnd'
class RewardHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    # the init is taken from the N+ implementation
    def _post_init(self):
        nn.init.normal_(self.reward.weight, std=(1.0 / np.sqrt(self.hidden_size + 1)))
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        return self.reward(hidden_states)

class GPT2RewardHead(nn.Module):
    def __init__(self, sft_model_path):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(sft_model_path)
        self.reward_head = RewardHead(self.llm.config)

    def forward(self, input_ids, attention_mask):
        transformer_outputs = self.llm.forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        last_hidden_state = transformer_outputs.hidden_states[-1]
        reward = self.reward_head(last_hidden_state).squeeze(-1) #2*bs, seq_len 
        # ic(reward.shape)
        return reward
    
model = GPT2RewardHead(sft_model_path)
# model.load_state_dict(torch.load("reward_model.pt"))
tokenizer = AutoTokenizer.from_pretrained(sft_model_path)


In [18]:
EPOCHS       = 1
LR           = 3e-6
WARMUP_STEPS = 0
grad_acc_steps = 8 # effective batch size 64
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_loader) * EPOCHS,
)

device = 'cuda'
model.llm.resize_token_embeddings(len(tokenizer))
model = model.to(device)

In [ ]:
# import wandb
# wandb.init(
#     project="gpt2_finetuning",
#     name=f"gpt2",
#     mode="online",
#     config={
#         "learning_rate": LR,
#         "epochs": EPOCHS,
#         "grad_accum": grad_acc_steps,
#         "device": device,
#         "dataset": "vwxyzjn/summarize_from_feedback_oai_preprocessing_1706381144",
#     }
# )

In [19]:
def get_reward(model, batch):

    query_responses = torch.cat(
        (
            batch["query_chosen_token"],
            batch["query_rejected_token"],
        ),
        dim=0,
    ).to(device)

    attention_mask = torch.cat(
        (
            batch["query_chosen_attention_mask"],
            batch["query_rejected_attention_mask"],
        ),
        dim=0,
    ).to(device)

    reward_logits = model(
        input_ids=query_responses,
        attention_mask=attention_mask,
    )
    # shape: [2*bs, seq_len]

    # last position where attention_mask == 1
    eos_idx = (
        attention_mask.shape[1]
        - 1
        - attention_mask.flip(1).argmax(dim=1)
    )

    rewards = reward_logits[
        torch.arange(
            reward_logits.size(0),
            device=reward_logits.device,
        ),
        eos_idx,
    ]

    return rewards

In [20]:

epoch_bar = tqdm(
    range(EPOCHS),
    desc="Training",
    position=0,
    leave=True
)
for epoch in range(EPOCHS):

    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        leave=False,
    )

    for step, batch in enumerate(train_bar):

        rewards = get_reward(model, batch) # this will return the 2*bs, reward one reward for each seq
        bs = batch["query_chosen_token"].shape[0]

        chosen_rewards = rewards[:bs]
        rejected_rewards = rewards[bs:]

        accuracy = (
            chosen_rewards > rejected_rewards
        ).float().mean()

        loss = -F.logsigmoid(
            chosen_rewards - rejected_rewards
        ).mean()

        total_loss += loss.item()

        # gradient accumulation
        loss = loss / grad_acc_steps
        loss.backward()

        if (step + 1) % grad_acc_steps == 0:

            nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        train_bar.set_postfix(
            loss=f"{loss.item() * grad_acc_steps:.4f}",
            acc=f"{accuracy.item():.4f}",
        )

        # wandb.log(
        #     {
        #         "loss_train": loss.item() * grad_acc_steps,
        #         "accuracy_train": accuracy.item(),
        #     }
        # )

    # handle leftover gradients
    if len(train_loader) % grad_acc_steps != 0:

        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    model.eval()

    eval_acc = []

    with torch.no_grad():

        eval_bar = tqdm(
            eval_loader,
            desc=f"Eval {epoch + 1}",
            leave=False,
        )

        for step, batch in enumerate(eval_bar):

            rewards = get_reward(model, batch)

            bs = batch["query_chosen_token"].shape[0]

            chosen_rewards = rewards[:bs]
            rejected_rewards = rewards[bs:]

            accuracy = (
                chosen_rewards > rejected_rewards
            ).float().mean()

            eval_acc.append(
                accuracy.item()
            )

            eval_bar.set_postfix(
                acc=f"{accuracy.item():.4f}"
            )

        mean_eval_acc = sum(eval_acc) / len(eval_acc)

        # wandb.log(
        #     {
        #         "eval_accuracy": mean_eval_acc,
        #         "epoch": epoch,
        #     }
        # )


    print(
        f"Epoch {epoch + 1} | "
        f"Train Loss: {total_loss / len(train_loader):.4f} | "
        f"Eval Acc: {mean_eval_acc:.4f}"
    )

Training:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/1:   0%|          | 0/11608 [00:00<?, ?it/s]

/media/system/ZERBUIS_EXT_STOR/temp/exp/rlhf/data_utils.py:190: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "query_token": torch.tensor(
/media/system/ZERBUIS_EXT_STOR/temp/exp/rlhf/data_utils.py:195: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "chosen_token": torch.tensor(
/media/system/ZERBUIS_EXT_STOR/temp/exp/rlhf/data_utils.py:200: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "rejected_token": torch.tensor(
/media/system/ZERBUIS_EXT_STOR/temp/exp/rlhf/data_utils.py:205: UserWarning: To copy construct from a tensor, it is recomme

Eval 1:   0%|          | 0/10476 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 0.6999 | Eval Acc: 0.5924


In [21]:
torch.save(model.state_dict(), "reward_model.pt")

In [22]:
model = GPT2RewardHead(sft_model_path)
model.load_state_dict(torch.load("reward_model.pt"))
model.eval()


GPT2RewardHead(
  (llm): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(50278, 768)
      (wpe): Embedding(1024, 768)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-11): 12 x GPT2Block(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D(nf=2304, nx=768)
            (c_proj): Conv1D(nf=768, nx=768)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D(nf=3072, nx=768)
            (c_proj): Conv1D(nf=768, nx=3072)
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_features=768, out_features